# Table Deals

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import re


Deals = pd.read_excel(
    '/content/drive/MyDrive/fin_project/not clean/Deals.xlsx',
    dtype={
        'Id': str,
        'Contact Name': str})
print(Deals.shape)

# City Column

In [ ]:
Deals['City'].value_counts().head(30)

In [ ]:
counts = Deals['City'].value_counts()
print((counts == 1).sum())  # how many values occur only once
print(counts[counts == 1].sum() / len(Deals))  # what share of rows

In [ ]:
# 1. How many rows are actually covered by the top-30 cities
top30_total = Deals['City'].value_counts().head(30).sum()
print(f"Top-30 City covers: {top30_total} rows ({top30_total/len(Deals):.1%})")

# 2. What the long tail actually looks like (values with frequency 1-3)
tail_values = Deals['City'].value_counts()
tail_values = tail_values[tail_values <= 3]
print(f"\nExamples of tail values (frequency ≤3):")
print(tail_values.head(40).index.tolist())

# 3. How many rows fall exactly into the tail
print(f"\nRows in the tail: {tail_values.sum()} ({tail_values.sum()/len(Deals):.1%})")

In [ ]:
# =========================================================
# CITY COLUMN PROCESSING — FINAL BLOCK FOR THE REPORT
# =========================================================
#
# SHORT SUMMARY FOR THE REPORT:
# --------------------------
# The City field is filled in for only 11.6% of deals (2509 of 21593).
# Analyzing fill rate by deal stage showed that this is not a
# random gap or a data error, but a systemic feature of the
# business process: City is filled in manually by the manager at
# the final stages of the funnel.
#
# Conclusion: City is an operational field for contract signing /
# material delivery, not a field collected at the lead-generation
# stage. So gaps at early stages are normal process behavior,
# not data loss.
#
# RECOMMENDATION FOR THE BUSINESS:
# If City is needed earlier in the funnel (e.g. for regional
# targeting at the lead stage), it should be added to the
# landing page / first-contact form, rather than left to the
# manager's discretion at the end of the process.
# =========================================================
# -----------------------------------------------------------
# STEP 1. Normalization
# -----------------------------------------------------------
Deals['City_clean'] = Deals['City'].fillna('Unknown').astype(str).str.strip()
missing_like = ['Unknown', '', '-', 'None', 'N/A', 'Na', 'Nan']
Deals['City_clean'] = Deals['City_clean'].replace(missing_like, 'Unknown')

# -----------------------------------------------------------
# STEP 2. Reference sets
# -----------------------------------------------------------
german_states = {
    'thüringen', 'bayern', 'sachsen', 'hessen', 'niedersachsen',
    'brandenburg', 'sachsen-anhalt', 'baden-württemberg',
    'nordrhein-westfalen', 'rheinland-pfalz', 'schleswig-holstein',
    'mecklenburg-vorpommern', 'saarland', 'bremen', 'hamburg', 'berlin'
}
countries_lower = {
    'germany', 'austria', 'switzerland', 'poland', 'france',
    'russia', 'ukraine', 'kazakhstan', 'netherlands', 'belgium'
}

# -----------------------------------------------------------
# STEP 3. Extract the city from addresses
# -----------------------------------------------------------
def try_extract_zip_city(segment: str):
    """Looks for 'ZIP City', e.g. '77761 Schiltach'."""
    match = re.search(r'\b\d{4,5}\s+([A-ZÄÖÜ][a-zäöüß\-]+)', segment)
    return match.group(1) if match else None


def extract_city(value: str) -> str:
    if value == 'Unknown':
        return 'Unknown'

    if ',' in value:
        parts = [p.strip() for p in value.split(',')]
        for part in parts:
            zip_city = try_extract_zip_city(part)
            if zip_city:
                return zip_city
        candidates = [
            p for p in parts
            if not re.search(r'\d', p)
            and p.lower() not in countries_lower
            and p.lower() not in german_states
        ]
        return candidates[0] if candidates else 'Address/Unknown'

    if re.search(r'\d', value):
        zip_city = try_extract_zip_city(value)
        return zip_city if zip_city else 'Address/Unknown'

    if value.lower() in countries_lower:
        return f'Country: {value}'

    return value


Deals['City_clean'] = Deals['City_clean'].apply(extract_city)
Deals['City_clean'] = Deals['City_clean'].apply(
    lambda x: x if x in ('Unknown', 'Address/Unknown') or x.startswith('Country:')
    else x.title()
)

# -----------------------------------------------------------
# STEP 4. Fill-rate diagnostics by Stage (for the report)
# -----------------------------------------------------------
city_fill_by_stage = (
    Deals.groupby('Stage')['City']
    .apply(lambda x: x.notna().mean())
    .sort_values(ascending=False)
)

# -----------------------------------------------------------
# STEP 5. Final check
# -----------------------------------------------------------
print(f"City fill rate: {Deals['City'].notna().mean():.1%}")
print(f"Unique City_clean values: {Deals['City_clean'].nunique()}")
print("\nCity fill rate by deal stage:")
print(city_fill_by_stage)

print("\nTop-20 cities after cleaning:")
print(Deals['City_clean'].value_counts().head(20))

# Sample size suitable for geo-analysis
geo_subset = Deals[~Deals['City_clean'].isin(['Unknown', 'Address/Unknown'])]
print(f"\nSample size for geo-analysis: {len(geo_subset)} rows "
      f"({len(geo_subset)/len(Deals):.1%} of the dataset)")

# Initial Amount Paid Column

In [ ]:
print(Deals['Initial Amount Paid'].dtype)
print(Deals['Initial Amount Paid'].unique()[:50])

In [ ]:
non_numeric = Deals['Initial Amount Paid'][
    Deals['Initial Amount Paid'].apply(lambda x: isinstance(x, str))
]
print(non_numeric.value_counts())

In [ ]:
def clean_amount(value):
    """
    Initial Amount Paid - convert to integers (Int64, with
    NaN support). The only non-numeric format in the data is
    '€ 3.500,00' (European notation: dot = thousands separator,
    comma = decimal separator).
    NaN is kept as missing (not filled with zero, since NaN
    means "amount not recorded", not "payment of 0").
    """
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (int, float)):
        return int(round(value))

    cleaned = str(value).replace('€', '').strip()
    cleaned = cleaned.replace('.', '')   # remove thousands separator
    cleaned = cleaned.replace(',', '.')  # comma -> decimal point
    return int(round(float(cleaned)))

Deals['Initial Amount Paid clean'] = (
    Deals['Initial Amount Paid']
    .apply(clean_amount)
    .astype('Int64')
)

# Check
print(Deals['Initial Amount Paid clean'].dtype)
print(f"Missing (NaN): {Deals['Initial Amount Paid clean'].isna().sum()}")
print(Deals[['Initial Amount Paid', 'Initial Amount Paid clean']].drop_duplicates().head(20))

# Offer Total Amount Column

In [ ]:
print(Deals['Offer Total Amount'].dtype)
print(Deals['Offer Total Amount'].unique()[:50])

In [ ]:
non_numeric_offer = Deals['Offer Total Amount'][
    Deals['Offer Total Amount'].apply(lambda x: isinstance(x, str))]
print(non_numeric_offer.value_counts())

In [ ]:
# clean_amount already exists above for Initial Amount Paid -
# reuse the same function, the format is the same (numbers + '€ X.XXX,XX' / '€ XXXXX,XX')

Deals['Offer Total Amount clean'] = (
    Deals['Offer Total Amount']
    .apply(clean_amount)
    .astype('Int64')
)

# Check
print(Deals['Offer Total Amount clean'].dtype)
print(f"Missing (NaN): {Deals['Offer Total Amount clean'].isna().sum()}")
print(Deals[['Offer Total Amount', 'Offer Total Amount clean']].drop_duplicates().head(25))

In [ ]:
mask_error = Deals['Initial Amount Paid clean'] > Deals['Offer Total Amount clean']
print(f"Rows with error (Initial > Offer): {mask_error.sum()}")
print(Deals.loc[mask_error, ['Initial Amount Paid clean', 'Offer Total Amount clean', 'Stage', 'Months of study']].head(20))

In [ ]:
diff = Deals.loc[mask_error, 'Initial Amount Paid clean'] - Deals.loc[mask_error, 'Offer Total Amount clean']
print(diff.value_counts())

# Level of Deutsch Column

In [ ]:
print(Deals['Level of Deutsch'].value_counts(dropna=False))

In [ ]:
pd.set_option('display.max_rows', 250)
print(Deals['Level of Deutsch'].value_counts(dropna=False))

In [ ]:
cyr_to_lat = {'а': 'a', 'б': 'b', 'в': 'b', 'с': 'c'}

def normalize_letter(ch):
    return cyr_to_lat.get(ch.lower(), ch.lower())

progress_keywords = [
    'waiting', 'awaits', 'taking exam', 'took exam', 'passed',
    'exam', 'result', 'studying', 'in progress', 'will', 'ongoing',
    'ready', 'finishing']
zero_level_keywords = [
    'did not study', 'has not studied', 'none', 'zero level', 'conversational']

def extract_level(raw):
    if pd.isna(raw):
        return (None, 'no_data')

    text = str(raw).strip()

    # obvious junk and placeholders
    if text in ('-', '?', '.', '0', '', 'None', 'none'):
        return ('Unknown', 'unclear')

    text_lower = text.lower()

    # no level at all -> A0
    if any(kw in text_lower for kw in zero_level_keywords):
        return ('A0', 'confirmed')

    # look for pattern: letter (Latin/Cyrillic) + digit 0-2, optional '+'
    # strip the '+' right away (round down to the base level)
    pattern = re.compile(r'([абвсabc])\s?([012])\s?(\+)?', re.IGNORECASE)
    matches = pattern.findall(text_lower)

    if not matches:
        return ('Unknown', 'unclear')

    # take the first match (current level), ignore the '+'
    letter, digit, _ = matches[0]
    level = f"{normalize_letter(letter).upper()}{digit}"

    status = 'in_progress' if any(
        kw in text_lower for kw in progress_keywords
    ) else 'confirmed'

    return (level, status)


# apply
extracted = Deals['Level of Deutsch'].apply(extract_level)
Deals['Level_clean'] = extracted.apply(lambda x: x[0])
Deals['Level_status'] = extracted.apply(lambda x: x[1])

# check
print("Level distribution:")
print(Deals['Level_clean'].value_counts(dropna=False))

print("\nStatus distribution:")
print(Deals['Level_status'].value_counts())

# share of unrecognized among filled rows
mask_filled = Deals['Level of Deutsch'].notna()
unclear = (Deals.loc[mask_filled, 'Level_clean'] == 'Unknown').mean()
print(f"\nShare of Unknown among filled rows: {unclear:.1%}")

# what exactly ended up as Unknown
mask_unclear = mask_filled & (Deals['Level_clean'] == 'Unknown')
print("\nWhat fell into Unknown:")
print(Deals.loc[mask_unclear, 'Level of Deutsch'].value_counts())

In [ ]:
mask_payment_done = (
    (Deals['Stage'] == 'Payment Done') &
    (Deals['Months of study'].notna()) &
    (Deals['Initial Amount Paid clean'] > Deals['Offer Total Amount clean']))

print(f"Rows with overpayment in Payment Done: {mask_payment_done.sum()}")
print(Deals.loc[mask_payment_done,
    ['Initial Amount Paid clean', 'Offer Total Amount clean', 'Months of study']
])

In [ ]:
print(Deals.loc[[12363, 15778],
    ['Initial Amount Paid clean', 'Offer Total Amount clean',
     'Payment Type', 'Product', 'Months of study']])

In [ ]:
print(Deals.loc[mask_payment_done & (diff == 500), 'Payment Type'].value_counts(dropna=False))

In [ ]:
# Look at all 21 rows and check what happens after the swap
print(Deals.loc[mask_payment_done,
    ['Initial Amount Paid clean', 'Offer Total Amount clean', 'Payment Type', 'Product']])

# Separately - if swapped, will Initial < Offer hold?
temp_initial = Deals.loc[mask_payment_done, 'Offer Total Amount clean']
temp_offer = Deals.loc[mask_payment_done, 'Initial Amount Paid clean']
print(f"\nAfter swap, all Initial < Offer: {(temp_initial < temp_offer).all()}")
print(f"After swap, all Initial <= Offer: {(temp_initial <= temp_offer).all()}")

In [ ]:
# Swap for all 21 rows in one command
initial_vals = Deals.loc[mask_payment_done, 'Initial Amount Paid clean'].copy()
offer_vals = Deals.loc[mask_payment_done, 'Offer Total Amount clean'].copy()

Deals.loc[mask_payment_done, 'Initial Amount Paid clean'] = offer_vals
Deals.loc[mask_payment_done, 'Offer Total Amount clean'] = initial_vals

# Check - should be 0 errors left
mask_check = (
    (Deals['Stage'] == 'Payment Done') &
    (Deals['Months of study'].notna()) &
    (Deals['Initial Amount Paid clean'] > Deals['Offer Total Amount clean'])
)
print(f"Remaining rows with Initial > Offer in Payment Done: {mask_check.sum()}")

# Date Columns

In [ ]:
print(Deals['Created Time'].dropna().head(10).tolist())
print()
print(Deals['Closing Date'].dropna().head(10).tolist())

In [ ]:
Deals['Created Time'] = pd.to_datetime(Deals['Created Time'], format='%d.%m.%Y %H:%M')
Deals['Closing Date'] = pd.to_datetime(Deals['Closing Date'], format='%d.%m.%Y')

# Check
print(Deals['Created Time'].dtype)
print(Deals['Closing Date'].dtype)
print()
print(Deals[['Created Time', 'Closing Date']].head(5))

In [ ]:
mask_date_error = (
    Deals['Closing Date'].notna() &
    (Deals['Closing Date'] < Deals['Created Time'].dt.normalize()))
print(f"Rows where Closing Date < Created Time: {mask_date_error.sum()}")

In [ ]:
print(Deals.loc[mask_date_error,
    ['Created Time', 'Closing Date', 'Stage', 'Months of study']].head(20))

In [ ]:
diff_days = (Deals.loc[mask_date_error, 'Created Time'].dt.normalize() -
             Deals.loc[mask_date_error, 'Closing Date']).dt.days

print("Distribution of the day difference (Closing Date earlier than Created Time):")
print(diff_days.describe())
print()
print("Rows with a difference > 30 days:")
print(diff_days[diff_days > 30].value_counts())
print()
print("Stage for these 44 rows:")
print(Deals.loc[mask_date_error, 'Stage'].value_counts())

In [ ]:
mask_pd_date_error = mask_date_error & (Deals['Stage'] == 'Payment Done')
print(Deals.loc[mask_pd_date_error,
    ['Created Time', 'Closing Date', 'Months of study',
     'Initial Amount Paid clean', 'Offer Total Amount clean']])

In [ ]:
Deals['Created Date'] = Deals['Created Time'].dt.normalize()
print(Deals[['Created Time', 'Created Date']].head(5))

In [ ]:
# Deal duration in days (from creation to closing)
Deals['Deal Duration Days'] = (Deals['Closing Date'] - Deals['Created Date']).dt.days

# Check
print(Deals['Deal Duration Days'].describe())
print()
# Negative values are exactly our 44 anomalies
print(f"Negative values: {(Deals['Deal Duration Days'] < 0).sum()}")
print(f"Zero values: {(Deals['Deal Duration Days'] == 0).sum()}")
print()
print(Deals[['Created Date', 'Closing Date', 'Deal Duration Days', 'Stage']].loc[Deals['Deal Duration Days'] < 0].head(10))

In [ ]:
mask_zero = Deals['Deal Duration Days'] == 0

print("Stage for zero-duration deals:")
print(Deals.loc[mask_zero, 'Stage'].value_counts())
print()
print("Lost Reason for zero-duration deals:")
print(Deals.loc[mask_zero, 'Lost Reason'].value_counts(dropna=False))

In [ ]:
# For Lost deals with negative duration - set to NaN
# (Closing Date for Lost deals is not meaningful under business rules)
mask_lost_negative = (Deals['Deal Duration Days'] < 0) & (Deals['Stage'] == 'Lost')
Deals.loc[mask_lost_negative, 'Deal Duration Days'] = None
print(f"Deal Duration Days cleared for Lost deals with negative duration: {mask_lost_negative.sum()}")

# Payment Done with negative duration
mask_pd_negative = (Deals['Deal Duration Days'] < 0) & (Deals['Stage'] == 'Payment Done')
print(f"\nPayment Done with negative duration: {mask_pd_negative.sum()}")
print(Deals.loc[mask_pd_negative, ['Created Date', 'Closing Date', 'Deal Duration Days', 'Months of study', 'Initial Amount Paid clean']])

In [ ]:
print(Deals['Deal Duration Days'].describe())
print()
print(Deals['Deal Duration Days'].value_counts().head(20))

In [ ]:
def classify_duration(days):
    if pd.isna(days):
        return 'Unknown'
    if days < 0:
        return 'Anomaly'
    if days == 0:
        return 'Same day'
    if days <= 7:
        return 'Fast (1-7 days)'
    if days <= 30:
        return 'Medium (8-30 days)'
    return 'Long (31+ days)'

Deals['Deal Duration Group'] = Deals['Deal Duration Days'].apply(classify_duration)

print(Deals['Deal Duration Group'].value_counts())
print()
# Breakdown by Stage within each group
print(pd.crosstab(Deals['Deal Duration Group'], Deals['Stage']))

# Month

In [ ]:
# Look at rows where Months of study = 0
mask_zero = Deals['Months of study'] == 0
print(f"Rows with 0 months: {mask_zero.sum()}")
print()
print(Deals.loc[mask_zero, ['Stage', 'Initial Amount Paid clean', 'Offer Total Amount clean', 'Months of study']].head(20))

In [ ]:
print(Deals['Months of study'].dtype)

In [ ]:
# Apply logic via masks
# First, fill everything as-is
Deals['Months of study'] = Deals['Months of study'].astype('Int64')

# Stage != Payment Done + NaN -> -1 (not a student)
mask_not_pd = (Deals['Stage'] != 'Payment Done') & (Deals['Months of study'].isna())
Deals.loc[mask_not_pd, 'Months of study'] = -1

# Stage = Payment Done + NaN -> 0 (paid, hasn't started yet)
mask_pd_nan = (Deals['Stage'] == 'Payment Done') & (Deals['Months of study'].isna())
Deals.loc[mask_pd_nan, 'Months of study'] = 0

# Stage = Payment Done + 0 -> 1 (started, month not yet accrued)
mask_pd_zero = (Deals['Stage'] == 'Payment Done') & (Deals['Months of study'] == 0)
Deals.loc[mask_pd_zero, 'Months of study'] = 1

# Check
print(Deals['Months of study'].value_counts().sort_index())

# Data Types

In [ ]:
# Convert Id and Contact Name to strings, for convenient joins
Deals['Id'] = Deals['Id'].astype('Int64').astype(str).replace('<NA>', None)
Deals['Contact Name'] = Deals['Contact Name'].astype('Int64').astype(str).replace('<NA>', None)

In [ ]:
# Course duration, inspect values
print(Deals['Course duration'].value_counts(dropna=False))

In [ ]:
# Convert Course duration to Int64
Deals['Course duration'] = Deals['Course duration'].astype('Int64')

# Check
print(Deals['Course duration'].dtype)
print(Deals['Course duration'].value_counts(dropna=False))

In [ ]:
print("Quality:")
print(Deals['Quality'].value_counts(dropna=False))
print()
print("Payment Type:")
print(Deals['Payment Type'].value_counts(dropna=False))
print()
print("Product:")
print(Deals['Product'].value_counts(dropna=False))
print()
print("Education Type:")
print(Deals['Education Type'].value_counts(dropna=False))

In [ ]:
print(Deals.loc[Deals['Quality'] == 'F', ['Stage', 'Source', 'Created Time']])

In [ ]:
#REF! in Education Type -> NaN (Excel formula error)
Deals['Education Type'] = Deals['Education Type'].replace('#REF!', None)
Deals['Education Type'] = Deals['Education Type'].replace({None: np.nan})

# Quality - leave F as is (don't touch)

# Check
print("Quality:")
print(Deals['Quality'].value_counts(dropna=False))
print()
print("Education Type:")
print(Deals['Education Type'].value_counts(dropna=False))

In [ ]:
# check for typos
print("Stage:")
print(Deals['Stage'].value_counts(dropna=False))
print()
print("Lost Reason:")
print(Deals['Lost Reason'].value_counts(dropna=False))

In [ ]:
Deals = Deals.drop(columns=['Initial Amount Paid', 'Offer Total Amount', 'Level of Deutsch', 'City'])
print(f"Columns remaining: {len(Deals.columns)}")
print(Deals.columns.tolist())

# SLA

In [ ]:
print(Deals['SLA'].value_counts(dropna=False).head(20))
print()
print(f"Unique values: {Deals['SLA'].nunique()}")
print(f"Example values: {Deals['SLA'].dropna().head(10).tolist()}")

In [ ]:
# Look at the actual data types in the column
Deals['SLA'].dropna().apply(type).value_counts()

In [ ]:
# timedelta to seconds
import datetime

def sla_to_seconds(val):
    if pd.isna(val):
        return None
    if isinstance(val, datetime.timedelta):
        return val.total_seconds()
    if isinstance(val, datetime.time):
        return val.hour * 3600 + val.minute * 60 + val.second
    return None

Deals['SLA_seconds'] = Deals['SLA'].apply(sla_to_seconds)

# Check
print(Deals['SLA_seconds'].describe())
print(f"\nMax SLA: {Deals['SLA_seconds'].max()/3600:.1f} hours")
print(f"Median SLA: {Deals['SLA_seconds'].median()/60:.1f} minutes")
print(f"Missing values: {Deals['SLA_seconds'].isna().sum()}")

In [ ]:
# Convert to hours for convenience
Deals['SLA_hours'] = Deals['SLA_seconds'] / 3600

print("SLA distribution in hours:")
print(Deals['SLA_hours'].describe())
print()

# How many rows have an abnormally large SLA (over 24h, 48h, a week)
print(f"SLA > 24 hours: {(Deals['SLA_hours'] > 24).sum()}")
print(f"SLA > 48 hours: {(Deals['SLA_hours'] > 48).sum()}")
print(f"SLA > 7 days: {(Deals['SLA_hours'] > 168).sum()}")
print(f"SLA > 30 days: {(Deals['SLA_hours'] > 720).sum()}")

In [ ]:
# Top-10 largest SLAs
print(Deals.nlargest(10, 'SLA_hours')[['SLA_hours', 'Stage', 'Created Time', 'Closing Date', 'Source']])
print()
# What Stage do rows with SLA > 30 days belong to
print(Deals.loc[Deals['SLA_hours'] > 720, 'Stage'].value_counts())

In [ ]:
# Do the anomalous SLAs match the timedelta type?

mask_timedelta = Deals['SLA'].dropna().apply(lambda x: isinstance(x, datetime.timedelta))
timedelta_indices = mask_timedelta[mask_timedelta].index

print(f"Rows with timedelta type: {len(timedelta_indices)}")
print(f"Rows with SLA > 24 hours: {(Deals['SLA_hours'] > 24).sum()}")
print()
print("SLA_hours for timedelta values:")
print(Deals.loc[timedelta_indices, 'SLA_hours'].describe())

The hypothesis was confirmed: all anomalous SLAs > 24 hours are exactly the timedelta values that Excel misinterpreted.
For example, 00:26:43 (26 minutes 43 seconds) was read as a timedelta in days, i.e. as 26 days 43 minutes — hence the inflated numbers.

In [ ]:
def sla_to_seconds_fixed(val):
    if pd.isna(val):
        return None
    if isinstance(val, datetime.timedelta):
        # Excel misread HH:MM:SS as days
        # total_seconds() gives days*86400 - keep only the remainder of the day
        total = val.total_seconds()
        # keep only the time-of-day portion (strip the days)
        return total % 86400
    if isinstance(val, datetime.time):
        return val.hour * 3600 + val.minute * 60 + val.second
    return None

Deals['SLA_seconds'] = Deals['SLA'].apply(sla_to_seconds_fixed)
Deals['SLA_hours'] = Deals['SLA_seconds'] / 3600

print(Deals['SLA_hours'].describe())
print(f"\nMax SLA: {Deals['SLA_hours'].max():.1f} hours")
print(f"Median SLA: {Deals['SLA_hours'].median()*60:.1f} minutes")
print(f"SLA > 24 hours after the fix: {(Deals['SLA_hours'] > 24).sum()}")

In [ ]:
print(Deals.loc[Deals['Stage'].isna(),
    ['Created Time', 'Source', 'Deal Owner Name', 'Initial Amount Paid clean']])

In [ ]:
# Drop rows where Stage = NaN (completely empty records)
Deals = Deals.dropna(subset=['Stage'])
print(f"Rows after removal: {len(Deals)}")

In [ ]:
# Drop the original SLA - we already have SLA_seconds and SLA_hours
# Also drop the original raw columns that have clean versions
cols_to_drop = ['SLA', 'Initial Amount Paid', 'Offer Total Amount',
                'City', 'Level of Deutsch']

Deals = Deals.drop(columns=[col for col in cols_to_drop if col in Deals.columns])
print(f"Dropped columns: {[col for col in cols_to_drop if col in Deals.columns]}")
print(f"Columns remaining: {len(Deals.columns)}")

# Payment Type

In [ ]:
mask_pd = (Deals['Stage'] == 'Payment Done') & (Deals['Initial Amount Paid clean'] > 0)
print(f"Payment Done with a payment: {mask_pd.sum()}")
print()
print("Current Payment Type for these rows:")
print(Deals.loc[mask_pd, 'Payment Type'].value_counts(dropna=False))
print()
print(f"Of which Payment Type is empty: {Deals.loc[mask_pd, 'Payment Type'].isna().sum()}")

In [ ]:
# Fill in Payment Type for Payment Done rows where it's empty
mask_to_fill = (
    (Deals['Stage'] == 'Payment Done') &
    (Deals['Initial Amount Paid clean'] > 0) &
    (Deals['Payment Type'].isna())
)

# One Payment: Initial == Offer (full payment)
mask_one_payment = mask_to_fill & (
    Deals['Initial Amount Paid clean'] == Deals['Offer Total Amount clean'])

# One Payment: Initial == Offer - 500 (difference of exactly 500 - treated as one-time)
mask_one_payment_500 = mask_to_fill & (
    Deals['Offer Total Amount clean'] - Deals['Initial Amount Paid clean'] == 500)

# Recurring Payments: Initial < Offer (installments)
mask_recurring = mask_to_fill & (
    Deals['Initial Amount Paid clean'] < Deals['Offer Total Amount clean']
    ) & ~mask_one_payment_500  # exclude the 500 difference

# Apply
Deals.loc[mask_one_payment, 'Payment Type'] = 'One Payment'
Deals.loc[mask_one_payment_500, 'Payment Type'] = 'One Payment'
Deals.loc[mask_recurring, 'Payment Type'] = 'Recurring Payments'

# Check
print(f"Filled as One Payment: {mask_one_payment.sum() + mask_one_payment_500.sum()}")
print(f"Filled as Recurring Payments: {mask_recurring.sum()}")
print()
print("Payment Type after filling:")
print(Deals.loc[mask_pd, 'Payment Type'].value_counts(dropna=False))

In [ ]:
# Look at the difference between Offer and Initial for the filled rows
diff_check = Deals.loc[mask_to_fill, 'Offer Total Amount clean'] - Deals.loc[mask_to_fill, 'Initial Amount Paid clean']
print("Distribution of the Offer - Initial difference:")
print(diff_check.value_counts().head(20))

# Revenue Calculation

The formula calculates how much money the school has already received from the student by the time the dataset was created.

*   **Block 1** — Installments, studying more than 1 month:

(Offer - Initial) / (Course duration - 1) * (Months of study - 1) + Initial
Offer - Initial → how much is left to pay after the first installment (e.g. 11000 - 1000 = 10000€)
/ (Course duration - 1) → divide by the remaining course months (11-1=10 months) → get the monthly payment (10000/10 = 1000€/month) * (Months of study - 1) → multiply by how many months the student has already paid after the first installment (e.g. 5-1=4 months → 4000€) + Initial → add the first installment (4000 + 1000 = 5000€)
Result: the school received 5000€ out of 11000€ — the student is in month 5 of 11.
* **Block 2** — Installments, studying 1 month:
Take only Initial — the student has only paid the first installment, no division needed.
* **Block 3** — One-time payment:
Take Initial — the student paid everything at once.


In [ ]:
def calc_revenue(row):
    initial = row['Initial Amount Paid clean']
    offer   = row['Offer Total Amount clean']
    months  = row['Months of study']
    course  = row['Course duration']
    payment = row['Payment Type']
    stage   = row['Stage']

    if stage != 'Payment Done':
        return 0

    if pd.isna(initial) or initial <= 0:
        return 0

    # One Payment - Initial only
    if payment == 'One Payment':
        return initial

    # Recurring Payments, 1 month - Initial only
    if payment == 'Recurring Payments' and months <= 1:
        return initial

    # Recurring Payments, more than 1 month:
    # Initial + (Offer - Initial) / (CourseDuration - 1) * (MonthsStudied - 1)
    if payment == 'Recurring Payments' and months > 1:
        if pd.isna(course) or (course - 1) == 0:
            return initial
        return initial + (offer - initial) / (course - 1) * (months - 1)

    return 0

Deals['Revenue'] = Deals.apply(calc_revenue, axis=1)

print(f"Total revenue: {Deals['Revenue'].sum():,.0f}€")
print()
print("Revenue by product:")
mask_pd = Deals['Stage'] == 'Payment Done'
print(Deals[mask_pd].groupby('Product')['Revenue'].sum().sort_values(ascending=False))

In [ ]:
Spend = pd.read_parquet('/content/drive/MyDrive/fin_project/Spend_clean.parquet')
print(f"Total ad spend: {Spend['Spend'].sum():,.0f}€")

# Saving to a Clean File

In [ ]:
# duplicates
mask_duplicate = Deals['Lost Reason'] == 'Duplicate'
mask_dup_pd = mask_duplicate & (Deals['Stage'] == 'Payment Done')
print(Deals.loc[mask_dup_pd, ['Stage', 'Source', 'Initial Amount Paid clean',
                               'Offer Total Amount clean', 'Months of study']])

In [ ]:
# Check business duplicates in Deals
mask_duplicate = Deals['Lost Reason'] == 'Duplicate'
print(f"Deals with Lost Reason = 'Duplicate': {mask_duplicate.sum()}")
print()
print("Distribution by Stage:")
print(Deals.loc[mask_duplicate, 'Stage'].value_counts())
print()
print("Distribution by Source:")
print(Deals.loc[mask_duplicate, 'Source'].value_counts().head(10))
print()
print("Example rows:")
print(Deals.loc[mask_duplicate, ['Stage', 'Source', 'Created Time', 'Lost Reason']].head(5))

These are real submissions from real people who simply filled in the form twice. The manager themselves flagged them as duplicates. These rows carry information: when the person submitted again, which source they came from the second time, and how quickly the manager spotted it.

Deleting them would lose that information. For example, the fact that 61% of duplicates come from Organic is an important business finding ("organic traffic generates a lot of repeat submissions, the form needs duplicate protection"). We kept the business duplicates in Deals because they are real events with valuable information — we simply exclude them from specific conversion calculations via a filter.

In [ ]:
# Save the cleaned dataset to the Drive folder
Deals.to_parquet('/content/drive/MyDrive/fin_project/Deals_clean.parquet', index=False)
print("Deals saved!")

In [ ]:
# Overall picture across all columns
summary = pd.DataFrame({
    'dtype': Deals.dtypes,
    'non_null': Deals.notna().sum(),
    'null': Deals.isna().sum(),
    'null_%': (Deals.isna().sum() / len(Deals) * 100).round(1),
    'unique': Deals.nunique()
})
print(summary)

# Descriptive Statistics

In [ ]:
# Check for duplicates in Deals
print(f"Duplicates across all columns: {Deals.duplicated().sum()}")
print(f"Duplicates by Id: {Deals.duplicated(subset=['Id']).sum()}")

In [ ]:
# Numeric columns
numeric_cols = ['Initial Amount Paid clean', 'Offer Total Amount clean',
                'Months of study', 'Course duration',
                'SLA_hours', 'Deal Duration Days']

for col in numeric_cols:
    print(f"{'='*40}")
    print(f"Column: {col}")
    print(f"  Mean:    {Deals[col].mean():.2f}")
    print(f"  Median:  {Deals[col].median():.2f}")
    print(f"  Mode:    {Deals[col].mode()[0]:.2f}")
    print(f"  Min:     {Deals[col].min():.2f}")
    print(f"  Max:     {Deals[col].max():.2f}")
    print(f"  Range:   {Deals[col].max() - Deals[col].min():.2f}")
    print(f"  Missing: {Deals[col].isna().sum()}")

In [ ]:
students = Deals[Deals['Months of study'] > 0]['Months of study']
print(f"Among students:")
print(f"  Mean:   {students.mean():.1f} months")
print(f"  Median: {students.median():.1f} months")
print(f"  Mode:   {students.mode()[0]:.1f} months")

In [ ]:
cat_cols = ['Stage', 'Quality', 'Source', 'Product',
            'Education Type', 'Payment Type', 'Level_clean']

for col in cat_cols:
    print(f"{'='*40}")
    print(f"Column: {col}")
    print(Deals[col].value_counts(dropna=False).head(10))